# Data Collection & Transcript Extraction
This notebook handles fetching YouTube video transcripts across three categories:
**Education**, **Tech/AI**, and **Entertainment/Media**. Transcripts are saved locally
as both `.json` (with timestamps) and `.txt` (plain text) for use in our RAG pipeline.

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
import json
import os

os.makedirs("../data/transcripts", exist_ok=True)
print("Directories ready")

Directories ready


## Helper Function
Fetches a transcript by video ID, saves both a timestamped JSON and a plain text version.

In [2]:
def get_transcript(video_id, video_title, category):
    try:
        ytt_api = YouTubeTranscriptApi()
        fetched = ytt_api.fetch(video_id)
        
        # Convert to list of dicts
        transcript = [{"text": s.text, "start": s.start, "duration": s.duration} 
                      for s in fetched]
        
        # Plain text
        text = " ".join([s["text"] for s in transcript])

        save_dir = f"../data/transcripts/{category}"
        os.makedirs(save_dir, exist_ok=True)

        with open(f"{save_dir}/{video_title}.json", "w") as f:
            json.dump({"video_id": video_id, "title": video_title,
                       "category": category, "transcript": transcript}, f, indent=2)

        with open(f"{save_dir}/{video_title}.txt", "w") as f:
            f.write(text)

        print(f"Saved [{category}]: {video_title} — {len(transcript)} segments")
        return text

    except Exception as e:
        print(f"Failed [{category}]: {video_title} — {e}")
        return None

## Video Dataset
A curated set of videos across Education, Tech/AI, and Entertainment/Media categories.
Video IDs are taken from the YouTube URL after `?v=`.

In [3]:
videos = [
    # Education
    {"id": "aircAruvnKk", "title": "neural_networks_explained", "category": "education"},
    {"id": "WUvTyaaNkzM", "title": "gradient_descent_explained", "category": "education"},
    {"id": "Ilg3gGewQ5U", "title": "transformers_explained", "category": "education"},

    # Tech / AI
    {"id": "hM_h0UA7upI", "title": "openai_gpt4_announcement", "category": "tech_ai"},
    {"id": "nAmC7SoVLd8", "title": "langchain_crash_course", "category": "tech_ai"},
    {"id": "qbIk7-JPB2c", "title": "rag_explained", "category": "tech_ai"},

    # Entertainment / Media
    {"id": "u-ZBLZATlhM", "title": "erwin_speech_aot", "category": "entertainment"},
    {"id": "HyqEtb5HE50", "title": "bad_bunny_hot_ones", "category": "entertainment"},
    {"id": "u37lG0BMFZk", "title": "fantano_igor_review", "category": "entertainment"},
]

transcripts = {}
for video in videos:
    transcripts[video["title"]] = get_transcript(video["id"], video["title"], video["category"])

Saved [education]: neural_networks_explained — 286 segments
Saved [education]: gradient_descent_explained — 243 segments
Saved [education]: transformers_explained — 193 segments
Saved [tech_ai]: openai_gpt4_announcement — 1489 segments
Saved [tech_ai]: langchain_crash_course — 1030 segments
Saved [tech_ai]: rag_explained — 1350 segments
Saved [entertainment]: erwin_speech_aot — 42 segments
Saved [entertainment]: bad_bunny_hot_ones — 360 segments
Saved [entertainment]: fantano_igor_review — 215 segments


## Validation
Check that all transcripts were saved correctly and review total segment counts.

In [4]:
for category in ["education", "tech_ai", "entertainment"]:
    path = f"../data/transcripts/{category}"
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith(".txt")]
        print(f"{category}: {len(files)} transcripts saved")
        for f in files:
            size = os.path.getsize(f"{path}/{f}")
            print(f"  - {f} ({size:,} bytes)")

education: 3 transcripts saved
  - neural_networks_explained.txt (18,430 bytes)
  - gradient_descent_explained.txt (15,797 bytes)
  - transformers_explained.txt (12,746 bytes)
tech_ai: 3 transcripts saved
  - rag_explained.txt (50,914 bytes)
  - langchain_crash_course.txt (36,394 bytes)
  - openai_gpt4_announcement.txt (56,915 bytes)
entertainment: 3 transcripts saved
  - fantano_igor_review.txt (15,786 bytes)
  - bad_bunny_hot_ones.txt (11,453 bytes)
  - erwin_speech_aot.txt (1,199 bytes)
